# MTG Commander — Mistral 7B LoRA Fine-Tuning

Fine-tunes `mistralai/Mistral-7B-Instruct-v0.2` on Commander deck examples using QLoRA (4-bit quantization).

**Before running:**
1. Set Runtime → Change runtime type → **T4 GPU** (free tier) or A100 (Colab Pro)
2. Upload your `chat_train.jsonl` and `chat_eval.jsonl` files from `training/data/processed/`
3. Add your Hugging Face token in the Secrets panel (key: `HF_TOKEN`)

> Mistral-7B-Instruct-v0.2 is a gated model — you must accept the license at https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2 first.

In [ ]:
# Install ML dependencies — numpy pinned to 1.x for torch/pyarrow compatibility
!pip install -q 'numpy<2.0'
!pip install -q transformers accelerate peft trl bitsandbytes huggingface_hub
print('Done! Run the restart cell next.')

In [ ]:
# RESTART RUNTIME — run this cell after install, then continue from the next cell
# Colab will show 'session crashed' — that is expected and intentional.
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print('Logged in to Hugging Face')

In [ ]:
# Upload your processed JSONL files
from google.colab import files
import os

os.makedirs('/content/data', exist_ok=True)
print('Upload chat_train.jsonl and chat_eval.jsonl from training/data/processed/')
uploaded = files.upload()

for fname, content in uploaded.items():
    dest = f'/content/data/{fname}'
    with open(dest, 'wb') as f:
        f.write(content)
    print(f'Saved {dest}')

TRAIN_FILE = '/content/data/chat_train.jsonl'
EVAL_FILE  = '/content/data/chat_eval.jsonl'

In [ ]:
# ---- Configuration (A100 80GB) ----
MODEL_ID      = 'mistralai/Mistral-7B-Instruct-v0.2'
OUTPUT_DIR    = '/content/mistral-commander-lora'
EPOCHS        = 5          # more epochs with larger dataset
LEARNING_RATE = 1e-4       # slightly lower LR for larger batches
BATCH_SIZE    = 8          # A100 80GB handles this comfortably
GRAD_ACCUM    = 2          # effective batch = 16
MAX_SEQ_LEN   = 4096       # full sequence length
LORA_R        = 64         # higher rank = more expressive adapter
LORA_ALPHA    = 128        # 2x rank
LORA_DROPOUT  = 0.05
USE_4BIT      = False      # A100 80GB: full bfloat16, no quantization needed
# ------------------------------------

In [ ]:
import json
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig
from trl import SFTTrainer
from datasets import load_dataset

# Quantization config for QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=USE_4BIT,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
) if USE_4BIT else None

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
print('Model loaded')

In [ ]:
def messages_to_text(record):
    messages = record.get('messages', [])
    if hasattr(tokenizer, 'apply_chat_template'):
        return {'text': tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}
    chunks = [f"{m['role'].upper()}: {m['content']}" for m in messages]
    return {'text': '\n\n'.join(chunks)}

data_files = {'train': TRAIN_FILE}
if EVAL_FILE:
    data_files['eval'] = EVAL_FILE

dataset = load_dataset('json', data_files=data_files)
train_ds = dataset['train'].map(messages_to_text)
eval_ds  = dataset['eval'].map(messages_to_text) if 'eval' in dataset else None

print(f'Train examples: {len(train_ds)}')
if eval_ds:
    print(f'Eval examples: {len(eval_ds)}')
print('Sample:', train_ds[0]['text'][:300])

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer
from transformers import TrainingArguments

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    logging_steps=5,
    save_strategy='epoch',
    eval_strategy='epoch' if eval_ds is not None else 'no',
    bf16=True,
    fp16=False,
    report_to='none',
    remove_unused_columns=False,
    warmup_steps=10,
    lr_scheduler_type='cosine',
)

import trl as _trl
_trl_new = tuple(int(x) for x in _trl.__version__.split(".")[:2]) >= (0, 9)
_tok_kwarg = {"processing_class": tokenizer} if _trl_new else {"tokenizer": tokenizer}
print(f"trl {_trl.__version__} — using {list(_tok_kwarg.keys())[0]}=")

trainer = SFTTrainer(
    model=model,
    **_tok_kwarg,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=training_args,
    peft_config=peft_config,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
)

trainer.train()

In [ ]:
import os
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'LoRA adapter saved to {OUTPUT_DIR}')
print('Files:', os.listdir(OUTPUT_DIR))

In [ ]:
# Zip and download the adapter
import shutil
from google.colab import files

zip_path = '/content/mistral-commander-lora.zip'
shutil.make_archive('/content/mistral-commander-lora', 'zip', OUTPUT_DIR)
files.download(zip_path)
print('Download started.')

In [ ]:
# Authenticate with HF write token (stored in Colab Secrets as HF_TOKEN_WRITE)
from google.colab import userdata
from huggingface_hub import login

hf_write_token = userdata.get('HF_TOKEN_WRITE')
if not hf_write_token:
    raise ValueError('HF_TOKEN_WRITE not found in Colab Secrets. '
                     'Go to the key icon in the left sidebar and add it.')

login(token=hf_write_token)
print('Logged in with write token successfully.')

In [ ]:
# OPTIONAL: Push adapter to your Hugging Face Hub repo
# trainer.model.push_to_hub('your-username/mistral-commander-lora')
# tokenizer.push_to_hub('your-username/mistral-commander-lora')

## Quick Inference Test
After training, test the adapter here before downloading.

In [ ]:
from peft import PeftModel

# Reload base model and merge adapter for inference
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
merged = PeftModel.from_pretrained(base, OUTPUT_DIR)

prompt = [
    {"role": "system", "content": "You are an expert Magic: The Gathering Commander deck builder. Given a commander and deck goal, produce a coherent 99-card decklist that matches the strategy."},
    {"role": "user", "content": "Commander: Kaalia of the Vast\nColor identity: W, B, R\nStrategy: Cheat Angels, Demons, and Dragons into play attacking. Aggressive and combo-capable.\nRespond with a JSON object containing a short description and the 99-card decklist."},
]

inputs = tokenizer.apply_chat_template(prompt, return_tensors='pt', add_generation_prompt=True).to('cuda')
with torch.no_grad():
    out = merged.generate(inputs, max_new_tokens=1024, temperature=0.7, do_sample=True)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

## Export Merged Model + GGUF for Local Use
Merges the LoRA adapter into the base model weights, then converts to GGUF Q4_K_M
format so you can run it locally with Ollama (works on AMD GPU via Vulkan).

In [ ]:
# Install llama.cpp Python bindings for GGUF conversion
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp 2>/dev/null || echo 'already cloned'
!pip install -q gguf

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MERGED_DIR = '/content/mistral-commander-merged'

print('Loading base model in bfloat16...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='cpu',  # merge on CPU to avoid VRAM pressure
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print('Loading LoRA adapter...')
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)

print('Merging adapter into base weights...')
model = model.merge_and_unload()

print(f'Saving merged model to {MERGED_DIR}...')
model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print('Done! Merged model saved.')

In [ ]:
import os

GGUF_PATH = '/content/mistral-commander-q4.gguf'

print('Converting to GGUF Q4_K_M (best quality/size tradeoff)...')
!python /content/llama.cpp/convert_hf_to_gguf.py \
    {MERGED_DIR} \
    --outfile /content/mistral-commander-fp16.gguf \
    --outtype f16

print('Quantizing to Q4_K_M...')
!/content/llama.cpp/build/bin/llama-quantize \
    /content/mistral-commander-fp16.gguf \
    {GGUF_PATH} \
    Q4_K_M

size_mb = os.path.getsize(GGUF_PATH) / (1024**2)
print(f'GGUF created: {GGUF_PATH} ({size_mb:.0f} MB)')

In [ ]:
# Build llama.cpp quantize tool (run this before convert_gguf if needed)
!apt-get install -qq cmake build-essential
!cd /content/llama.cpp && cmake -B build -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release -DGGML_NATIVE=OFF . && cmake --build build --target llama-quantize -j4
print('llama.cpp built!')

In [ ]:
from google.colab import files
import os

GGUF_PATH = '/content/mistral-commander-q4.gguf'

size_mb = os.path.getsize(GGUF_PATH) / (1024**2)
print(f'Downloading {size_mb:.0f} MB GGUF file...')
print('This will take a moment — the file is ~4GB')
files.download(GGUF_PATH)